# Strategy Backtest v3 — rolling walk-forward (anti-overfitting pass)

The v2 "out-of-sample" year (2025-09 → 2026-09) has been looked at, so it can no longer validate new tuning. v3 therefore:

1. Extends history back to **2020-06** (Alpaca daily bars) so signals exist from ~2021-03.
2. Uses a **rolling walk-forward**: parameters/choices are picked on the trailing 12 months, traded for the next 3 months,
   rolled quarterly from **2022-04-01**; results are stitched into one out-of-sample track.
3. Reports three segments: the stitched walk-forward period, the **never-seen** part (2022-04-01 → 2024-09-16: no rule or
   parameter in this project was chosen on it) and the already-seen part (2024-09-17 → latest).
4. Keeps the candidate list **small and pre-declared** (below). Same engine as v2: point-in-time features, decision at the close,
   fill at the next open, 0.1% cost per side.

**Pre-declared candidates** (all variations of the live strategy C0 = weekly top-10, score 0.5·Technical + 0.5·RS, inverse-vol,
max 4 per sector, score > 0):

| id | idea | fixed setting |
|---|---|---|
| C1 | rank buffer | keep a holding while its rank ≤ 15 |
| C2 | 12-1 momentum | 0.5·Technical + 0.5·rank(12-1m momentum) |
| C3 | sector-neutral momentum | 0.5·Technical + 0.5·rank(12-1m momentum − sector ETF's) |
| C4 | volatility-adjusted RS | RS components divided by own volatility |
| C5 | equal weight | instead of inverse-vol |
| C6 | soft regime | exposure × 0.5 when QQQ < 200-day MA (at rebalances) |
| C7 | portfolio vol target | scale to 20% annualized (21-day realized) |
| C8 | ATR stop | exit holding at peak − 3×ATR, re-enter only at a rebalance |
| C9 | QQQ core | 50% QQQ + 50% C0, rebalanced weekly |
| C10 | monthly rebalance | last session of the month |
| C11 | buffer + QQQ core | C1 and C9 combined |
| WF-P | walk-forward params | each quarter pick N ∈ {5,10} × w_tech ∈ {0.3,0.5,0.7} by trailing-12m Sharpe |
| WF-M | walk-forward meta | each quarter pick the best of C0–C6, C8–C11 by trailing-12m Sharpe (diagnostic only) |

**Pre-declared switch rule:** a candidate replaces C0 live only if, on the stitched walk-forward track, its Sharpe is higher AND its
max drawdown is better (or its Sharpe is higher with max DD within 2 points), AND its Sharpe on the never-seen segment is not
below C0's. WF-M is diagnostic (not deployable as-is). Among qualifiers, the highest stitched Sharpe wins.

**Bias warning:** the 78-stock universe was hand-picked in 2026 with hindsight; this matters even more for 2022–2023 (stocks were
chosen partly because they later rose). Late listings are simply ineligible until they have 200 bars. SPY/QQQ are the only
bias-free benchmarks; compare candidates mainly against C0 (same bias).

### Setup
Load cached bars and scores; walk-forward windows are defined in the next cell.

In [1]:

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import backtest_engine as be

pd.set_option("display.max_rows", 200, "display.max_columns", 50, "display.width", 250)
LONG_START = "2020-06-01"
bars, dropped_partial = be.load_bars(refresh=True, cache_name="bars_daily_long.pkl", start=LONG_START)
tech = be.build_technical(bars)
close_all, open_all = be.wide(bars, "Close"), be.wide(bars, "Open")
syms = [s for s in be.TRADABLE if s in close_all.columns]
C = close_all[syms]
idx = close_all.index

score_tech = be.wide(tech, "Technical_Score").reindex(index=idx, columns=syms)
elig = be.bool_wide(tech, "eligible", idx, syms)
atr = be.wide(tech, "atr").reindex(index=idx, columns=syms)
rs, _ = be.relative_strength(close_all, syms)
rs_va, _ = be.relative_strength(close_all, syms, vol_adjust=True)
mom = be.momentum_score(close_all, syms)
mom_sn = be.momentum_score(close_all, syms, sector_neutral=True)
vol63 = C.pct_change(fill_method=None).rolling(63).std()
regime_qqq = be.regime_series(close_all, "QQQ")
weekly = be.weekly_rebalance_days(idx)
monthly = be.monthly_rebalance_days(idx)
print(f"Bars {idx.min():%Y-%m-%d} → {idx.max():%Y-%m-%d} | partial bar dropped: {dropped_partial} | "
      f"first eligible date: {elig.any(axis=1).idxmax():%Y-%m-%d} | eligible names at WF start: {int(elig.loc['2022-04-01':].iloc[0].sum())}")

Bars 2020-06-01 → 2026-09-24 | partial bar dropped: False | first eligible date: 2021-03-16 | eligible names at WF start: 65


### Candidate builders and walk-forward machinery

In [2]:
WF_START = "2022-04-01"
SEEN_START = "2024-09-17"
SEGMENTS = {"WF stitched": (WF_START, None), "Never-seen 2022-04→2024-09": (WF_START, "2024-09-16"),
            "Seen 2024-09→now": (SEEN_START, None)}


def ranked(score, n=10, **kw):
    reb = kw.pop("rebalance_days", weekly)
    return be.rank_targets(score, elig, vol63, n=n, rebalance_days=reb, **kw), reb


def full(t):
    return t.reindex(index=idx, columns=close_all.columns).fillna(0.0)


def base_score(w=0.5, rel=rs):
    return w * score_tech + (1 - w) * rel


CANDIDATES = {}
CANDIDATES["C0 current (top10, w=0.5, weekly)"] = (*ranked(base_score()), {})
CANDIDATES["C1 rank buffer 15"] = (*ranked(base_score(), buffer_rank=15), {})
CANDIDATES["C2 12-1 momentum"] = (*ranked(base_score(rel=mom)), {})
CANDIDATES["C3 sector-neutral momentum"] = (*ranked(base_score(rel=mom_sn)), {})
CANDIDATES["C4 vol-adjusted RS"] = (*ranked(base_score(rel=rs_va)), {})
CANDIDATES["C5 equal weight"] = (*ranked(base_score(), vol_sizing=False), {})
CANDIDATES["C6 soft regime (QQQ<200d -> 50%)"] = (*ranked(base_score(), regime=regime_qqq, regime_scale=0.5), {})
c0_t, _ = ranked(base_score())
CANDIDATES["C7 vol target 20%"] = (c0_t, weekly, {"vol_target": (0.20, 21)})
CANDIDATES["C8 ATR 3x stop"] = (be.atr_stop_overlay(c0_t, C, atr, weekly, k=3.0), weekly, {})
CANDIDATES["C9 50% QQQ + 50% C0"] = (be.blend_with_core(full(c0_t), "QQQ", 0.5, list(close_all.columns)), weekly, {})
CANDIDATES["C10 monthly rebalance"] = (*ranked(base_score(), rebalance_days=monthly), {})
c1_t, _ = ranked(base_score(), buffer_rank=15)
CANDIDATES["C11 buffer15 + 50% QQQ core"] = (be.blend_with_core(full(c1_t), "QQQ", 0.5, list(close_all.columns)), weekly, {})
CANDIDATES = {k: (full(t), r, kw) for k, (t, r, kw) in CANDIDATES.items()}


def sim(target, reb, kw, s, e=None):
    return be.simulate(open_all, close_all, target, s, e, rebalance=reb, **kw)


def quarter_starts(start, end):
    return [d for d in pd.date_range(start, end, freq="QS")]


def walk_forward(pool, train_months=12):
    """Each quarter: pick the pool member with the best trailing-12m Sharpe, trade it for the quarter."""
    starts = quarter_starts(WF_START, idx[-1])
    target = pd.DataFrame(0.0, index=idx, columns=close_all.columns)
    reb = pd.Series(False, index=idx)
    picks = []
    for i, qs in enumerate(starts):
        tr_s, tr_e = qs - pd.DateOffset(months=train_months), qs - pd.Timedelta(days=1)
        scores = {k: be.metrics(sim(*pool[k], tr_s, tr_e))["Sharpe"] for k in pool}
        best = max(scores, key=lambda k: -np.inf if pd.isna(scores[k]) else scores[k])
        lo = idx.searchsorted(qs) - 1                                   # decision on the close before the quarter
        hi = idx.searchsorted(starts[i + 1]) - 1 if i + 1 < len(starts) else len(idx)
        t, r, _ = pool[best]
        target.iloc[lo:hi] = t.iloc[lo:hi].values
        reb.iloc[lo:hi] = r.reindex(idx).astype("boolean").fillna(False).astype(bool).iloc[lo:hi].values
        reb.iloc[lo] = True                                             # switch -> full rebalance
        picks.append({"quarter": qs.date(), "pick": best, "train Sharpe": round(scores[best], 2)})
    return target, reb, pd.DataFrame(picks)


# WF-P: parameter walk-forward over N x w_tech
param_pool = {f"N={n}, w={w}": (full(ranked(base_score(w), n=n)[0]), weekly, {}) for n in [5, 10] for w in [0.3, 0.5, 0.7]}
wfp_t, wfp_r, wfp_picks = walk_forward(param_pool)
# WF-M: meta walk-forward over the fixed candidates (vol-target C7 excluded: its path-dependent scaling cannot be stitched)
meta_pool = {k: v for k, v in CANDIDATES.items() if not k.startswith("C7")}
wfm_t, wfm_r, wfm_picks = walk_forward(meta_pool)
CANDIDATES["WF-P walk-forward N & w_tech"] = (wfp_t, wfp_r, {})
CANDIDATES["WF-M walk-forward meta (diagnostic)"] = (wfm_t, wfm_r, {})
print("WF-P picks:\n", wfp_picks.to_string(index=False))
print("WF-M picks:\n", wfm_picks.to_string(index=False))

WF-P picks:
    quarter        pick  train Sharpe
2022-04-01  N=5, w=0.3          0.95
2022-07-01  N=5, w=0.3          0.25
2022-10-01  N=5, w=0.3         -0.26
2023-01-01 N=10, w=0.5         -0.16
2023-04-01 N=10, w=0.3         -0.23
2023-07-01 N=10, w=0.7          0.92
2023-10-01 N=10, w=0.3          0.81
2024-01-01 N=10, w=0.3          1.33
2024-04-01 N=10, w=0.3          2.02
2024-07-01  N=5, w=0.3          1.73
2024-10-01 N=10, w=0.7          2.17
2025-01-01 N=10, w=0.5          2.67
2025-04-01 N=10, w=0.5          1.42
2025-07-01 N=10, w=0.5          2.09
2025-10-01 N=10, w=0.5          2.59
2026-01-01 N=10, w=0.5          1.63
2026-04-01 N=10, w=0.5          2.68
2026-07-01  N=5, w=0.7          2.47
WF-M picks:
    quarter                             pick  train Sharpe
2022-04-01            C10 monthly rebalance          1.16
2022-07-01            C10 monthly rebalance          0.21
2022-10-01                 C2 12-1 momentum         -0.13
2023-01-01                   C8 ATR 3x 

### Results on the stitched walk-forward track and its segments

In [3]:
rows, curves = [], {}
bench = {}
for b in ["QQQ", "SPY"]:
    t = pd.DataFrame(0.0, index=idx, columns=close_all.columns)
    t[b] = 1.0
    bench[f"{b} buy & hold"] = (t, None, {})
for name, (t, r, kw) in {**bench, **CANDIDATES}.items():
    for seg, (s, e) in SEGMENTS.items():
        res = sim(t, r, kw, s, e)
        rows.append({**be.metrics(res, name), "Segment": seg})
        if seg == "WF stitched":
            curves[name] = res["equity"]
for seg, (s, e) in SEGMENTS.items():   # equal-weight universe, bought at segment start, never rebalanced
    t, alive = be.buy_and_hold_target(close_all, syms, s)
    res = be.simulate(open_all, close_all, t, s, e)
    rows.append({**be.metrics(res, "EW universe buy & hold"), "Segment": seg})
    if seg == "WF stitched":
        curves["EW universe buy & hold"] = res["equity"]
res_df = pd.DataFrame(rows)
res_df.round(4).to_csv(be.REPORTS_DIR / "strategy_comparison_v3.csv", index=False)

COLS = ["CAGR %", "Total Return %", "Sharpe", "Sortino", "Calmar", "Max DD %", "Exposure %", "Turnover x/yr", "Trades",
        "Win Rate % (closed, net)"]
view = res_df.pivot_table(index="Strategy", columns="Segment", values=["Sharpe", "Max DD %", "CAGR %"], sort=False).round(2)
wf = res_df[res_df["Segment"] == "WF stitched"].set_index("Strategy")[COLS].round(2).sort_values("Sharpe", ascending=False)
wf
view

,CAGR %,Total Return %,Sharpe,Sortino,Calmar,Max DD %,Exposure %,Turnover x/yr,Trades,"Win Rate % (closed, net)"
Strategy,,,,,,,,,,
C6 soft regime (QQQ<200d -> 50%),46.57,450.36,1.47,2.23,1.63,-28.58,87.79,39.84,849,46.13
WF-M walk-forward meta (diagnostic),53.96,585.31,1.46,2.21,1.36,-39.60,99.56,33.23,687,44.23
C7 vol target 20%,38.63,329.28,1.42,2.14,1.48,-26.16,85.06,37.51,849,46.01
"C0 current (top10, w=0.5, weekly)",46.84,454.83,1.40,2.10,1.42,-32.96,100.00,43.57,849,46.48
C5 equal weight,60.66,728.83,1.40,2.14,1.53,-39.61,100.00,38.85,849,47.79
C8 ATR 3x stop,43.52,401.13,1.36,2.04,1.26,-34.67,96.82,45.34,899,45.67
C9 50% QQQ + 50% C0,32.73,253.52,1.28,1.89,1.18,-27.64,100.00,22.24,850,46.72
C4 vol-adjusted RS,34.65,277.02,1.23,1.84,1.18,-29.26,99.92,42.05,847,45.76
C10 monthly rebalance,39.45,340.69,1.23,1.82,1.31,-30.22,100.00,15.50,330,46.88


Sharpe                                                Max DD %                                                  CAGR %                                            
Segment                             WF stitched Never-seen 2022-04→2024-09 Seen 2024-09→now WF stitched Never-seen 2022-04→2024-09 Seen 2024-09→now WF stitched Never-seen 2022-04→2024-09 Seen 2024-09→now
Strategy                                                                                                                                                                                                   
QQQ buy & hold                             0.85                       0.61             1.14      -29.07                     -29.07           -22.77       18.11                      12.23            25.23
SPY buy & hold                             0.85                       0.67             1.08      -21.29                     -21.29           -18.75       14.03                      10.74            17.86
C0 current (top10, w=0.5, weekly)          1.40                       0.81             1.98      -32.96                     -22.33           -32.96       46.84                      20.45            85.51
C1 rank buffer 15                          1.16                       0.60             1.71      -35.02                     -24.66           -35.02       36.55                      13.44            71.14
C2 12-1 momentum                           1.18                       1.23             1.19      -39.58                     -20.83           -39.58       42.72                      33.65            53.60
C3 sector-neutral momentum                 1.17                       1.35             1.09      -42.74                     -19.71           -42.74       41.82                      37.75            46.01
C4 vol-adjusted RS                         1.23                       0.71             1.79      -29.26                     -22.34           -29.26       34.65                      15.96            60.97
C5 equal weight                            1.40                       0.83             1.98      -39.61                     -28.58           -39.61       60.66                      26.69           112.91
C6 soft regime (QQQ<200d -> 50%)           1.47                       0.92             1.98      -28.58                     -22.33           -28.58       46.57                      22.28            81.41
C7 vol target 20%                          1.42                       0.75             2.11      -26.16                     -21.02           -26.16       38.63                      16.03            71.03
C8 ATR 3x stop                             1.36                       0.87             1.84      -34.67                     -21.95           -34.67       43.52                      21.89            73.82
C9 50% QQQ + 50% C0                        1.28                       0.78             1.82      -27.64                     -23.96           -27.64       32.73                      16.86            54.11
C10 monthly rebalance                      1.23                       1.08             1.39      -30.22                     -18.98           -30.22       39.45                      30.61            50.91
C11 buffer15 + 50% QQQ core                1.13                       0.66             1.65      -28.69                     -24.20           -28.69       28.04                      13.40            48.15
WF-P walk-forward N & w_tech               1.14                       0.38             1.97      -33.32                     -33.32           -31.69       36.49                       7.16            81.94
WF-M walk-forward meta (diagnostic)        1.46                       1.24             1.67      -39.60                     -17.66           -39.60       53.96                      36.50            77.11
EW universe buy & hold                     0.86                       0.69             1.23      -35.76                     -35.76           -37.14       27.80      

### Decision (pre-declared rule)

In [5]:
seg_sharpe = res_df.pivot_table(index="Strategy", columns="Segment", values="Sharpe")
seg_dd = res_df.pivot_table(index="Strategy", columns="Segment", values="Max DD %")
BASE = "C0 current (top10, w=0.5, weekly)"
b_sh, b_dd = seg_sharpe.at[BASE, "WF stitched"], seg_dd.at[BASE, "WF stitched"]
b_unseen = seg_sharpe.at[BASE, "Never-seen 2022-04→2024-09"]
qual = []
for name in CANDIDATES:
    if name == BASE or name.startswith("WF-M"):
        continue
    sh, dd, un = seg_sharpe.at[name, "WF stitched"], seg_dd.at[name, "WF stitched"], seg_sharpe.at[name, "Never-seen 2022-04→2024-09"]
    ok = sh > b_sh and (dd > b_dd or dd >= b_dd - 2.0) and un >= b_unseen
    qual.append({"Candidate": name, "WF Sharpe": round(sh, 2), "WF MaxDD %": round(dd, 1), "Never-seen Sharpe": round(un, 2),
                 "Qualifies": ok})
qual = pd.DataFrame(qual).sort_values("WF Sharpe", ascending=False)
winners = qual[qual["Qualifies"]]
DECISION = winners.iloc[0]["Candidate"] if len(winners) else BASE
q = seg_sharpe.loc["QQQ buy & hold", "WF stitched"]
print(f"C0 stitched: Sharpe {b_sh:.2f}, MaxDD {b_dd:.1f}%, never-seen Sharpe {b_unseen:.2f} | QQQ stitched Sharpe {q:.2f}")
print(f"DECISION: {'switch to ' + DECISION if DECISION != BASE else 'keep C0 (no candidate met the rule)'}")
qual

C0 stitched: Sharpe 1.40, MaxDD -33.0%, never-seen Sharpe 0.81 | QQQ stitched Sharpe 0.85
DECISION: switch to C6 soft regime (QQQ<200d -> 50%)


,Candidate,WF Sharpe,WF MaxDD %,Never-seen Sharpe,Qualifies
5,C6 soft regime (QQQ<200d -> 50%),1.47,-28.60,0.92,True
6,C7 vol target 20%,1.42,-26.20,0.75,False
4,C5 equal weight,1.40,-39.60,0.83,False
7,C8 ATR 3x stop,1.36,-34.70,0.87,False
8,C9 50% QQQ + 50% C0,1.28,-27.60,0.78,False
3,C4 vol-adjusted RS,1.23,-29.30,0.71,False
9,C10 monthly rebalance,1.23,-30.20,1.08,False
1,C2 12-1 momentum,1.18,-39.60,1.23,False
2,C3 sector-neutral momentum,1.17,-42.70,1.35,False
0,C1 rank buffer 15,1.16,-35.00,0.60,False


### Equity curves

In [6]:
show = list(dict.fromkeys([BASE, DECISION, "C9 50% QQQ + 50% C0", "WF-P walk-forward N & w_tech", "QQQ buy & hold",
                           "SPY buy & hold", "EW universe buy & hold"]))
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
for n in show:
    eq = curves[n]
    style = dict(lw=2.6) if n == DECISION else dict(lw=1.3, ls="--" if "buy & hold" in n else "-")
    ax1.plot(eq.index, eq.values, label=f"{n} ({(eq.iloc[-1] - 1) * 100:+.0f}%, Sharpe {seg_sharpe.at[n, 'WF stitched']:.2f})", **style)
    ax2.plot(eq.index, (eq / eq.cummax().clip(lower=1) - 1) * 100, lw=1.1 if n != DECISION else 2, **({} if n != DECISION else {"color": "k"}))
for ax in (ax1, ax2):
    ax.axvspan(pd.Timestamp(WF_START), pd.Timestamp(SEEN_START), color="#e0f2fe", alpha=0.5)
    ax.grid(alpha=0.3)
ax1.set_yscale("log")
ax1.set_ylabel("Equity (log, start = 1.0)")
ax1.set_title("Walk-forward v3: stitched out-of-sample track 2022-04 → now (blue band = never-seen segment). "
              "Universe hand-picked → selection bias", fontsize=10)
ax1.legend(fontsize=8, loc="upper left")
ax2.set_ylabel("Drawdown %")
fig.tight_layout()
CHART = be.REPORTS_DIR / "strategy_walkforward_v3.png"
fig.savefig(CHART, dpi=130)
print(f"Saved {CHART.name} and strategy_comparison_v3.csv")

Saved strategy_walkforward_v3.png and strategy_comparison_v3.csv
